In [ ]:
# ===============================
# RANDOM FOREST REGRESSION
# ===============================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2. LOAD DATA
df = pd.read_csv("../data/rf_regression.csv")

print("First 5 rows:")
print(df.head())
print()

print("Dataset shape:")
print(df.shape)
print()

print("Missing values:")
print(df.isnull().sum())
print()

# 3. DEFINE FEATURES AND TARGET
X = df.drop("price_k", axis=1)
y = df["price_k"]

# 4. SPLIT DATA: 70% TRAIN, 15% VALIDATION, 15% TEST

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print()

# 5. IDENTIFY NUMERIC AND CATEGORICAL COLUMNS
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print()

# 6. HANDLE MISSING VALUES STEP BY STEP

# Numeric imputation
num_imputer = SimpleImputer(strategy="median")

X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index
)

X_val_num = pd.DataFrame(
    num_imputer.transform(X_val[numeric_features]),
    columns=numeric_features,
    index=X_val.index
)

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[numeric_features]),
    columns=numeric_features,
    index=X_test.index
)

# Categorical imputation
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train[categorical_features]),
    columns=categorical_features,
    index=X_train.index
)

X_val_cat = pd.DataFrame(
    cat_imputer.transform(X_val[categorical_features]),
    columns=categorical_features,
    index=X_val.index
)

X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test[categorical_features]),
    columns=categorical_features,
    index=X_test.index
)

# 7. ENCODING
# Fit encoder ONLY on training categorical data
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat_encoded = encoder.fit_transform(X_train_cat)
X_val_cat_encoded = encoder.transform(X_val_cat)
X_test_cat_encoded = encoder.transform(X_test_cat)

encoded_feature_names = encoder.get_feature_names_out(categorical_features)

X_train_cat_encoded = pd.DataFrame(
    X_train_cat_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

X_val_cat_encoded = pd.DataFrame(
    X_val_cat_encoded,
    columns=encoded_feature_names,
    index=X_val.index
)

X_test_cat_encoded = pd.DataFrame(
    X_test_cat_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

# 8. COMBINE NUMERIC + ENCODED CATEGORICAL FEATURES
X_train_final = pd.concat([X_train_num, X_train_cat_encoded], axis=1)
X_val_final = pd.concat([X_val_num, X_val_cat_encoded], axis=1)
X_test_final = pd.concat([X_test_num, X_test_cat_encoded], axis=1)

print("Final processed shapes:")
print("X_train_final:", X_train_final.shape)
print("X_val_final  :", X_val_final.shape)
print("X_test_final :", X_test_final.shape)
print()

# 9. DEFINE RANDOM FOREST REGRESSOR
rf = RandomForestRegressor(random_state=42)

# 10. DEFINE PARAMETER DISTRIBUTIONS FOR RANDOMIZED SEARCH
param_distributions = {
    "n_estimators": [100, 150, 200, 300],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

# 11. RANDOMIZED SEARCH
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,              # change to 15 or 20 if your computer is fine
    scoring="r2",           # good main metric for regression tuning
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=1,
    refit=True
)

# 12. FIT ON TRAINING DATA ONLY
random_search.fit(X_train_final, y_train)

print("\nBest Parameters:")
print(random_search.best_params_)

print("\nBest Cross-Validation R²:")
print(random_search.best_score_)

# 13. GET BEST MODEL
best_model = random_search.best_estimator_

# 14. EVALUATION FUNCTION
def evaluate_regression(model, X_data, y_data, dataset_name):
    y_pred = model.predict(X_data)

    mae = mean_absolute_error(y_data, y_pred)
    rmse = np.sqrt(mean_squared_error(y_data, y_pred))
    r2 = r2_score(y_data, y_pred)

    print(f"\n{'='*50}")
    print(f"{dataset_name} RESULTS")
    print(f"{'='*50}")
    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²  :", round(r2, 4))


First 5 rows:
     brand fuel_type transmission seller_type  car_age_years  mileage_km  \
0  Hyundai    Hybrid       Manual      Dealer            3.6      103461   
1   Toyota    Diesel    Automatic      Dealer            9.9       91959   
2    Honda    Diesel       Manual  Individual            4.3       87872   
3    Honda    Hybrid    Automatic   Certified            8.2      182724   
4   Toyota  Electric    Automatic      Dealer            3.5      102939   

   engine_size_l  horsepower  previous_owners  price_k  
0            1.6       145.0                1    16.00  
1            2.0       128.0                1    14.66  
2            1.0        75.0                2    11.94  
3            1.4       154.0                2    13.04  
4            1.7       172.0                1    28.82  

Dataset shape:
(10000, 10)

Missing values:
brand              100
fuel_type          100
transmission         0
seller_type          0
car_age_years        0
mileage_km           0
engi

In [5]:
# 16. EVALUATE ON TRAINING SET
evaluate_regression(best_model, X_train_final, y_train, "TRAINING SET")


TRAINING SET RESULTS
MAE : 0.675
RMSE: 0.8636
R²  : 0.9831


In [6]:
# 17. EVALUATE ON VALIDATION SET
evaluate_regression(best_model, X_val_final, y_val, "VALIDATION SET")


VALIDATION SET RESULTS
MAE : 1.8411
RMSE: 2.3051
R²  : 0.8772


In [7]:
# 18. EVALUATE ON TEST SET
evaluate_regression(best_model, X_test_final, y_test, "TEST SET")



TEST SET RESULTS
MAE : 1.7637
RMSE: 2.2447
R²  : 0.8814


In [8]:
# 19. FEATURE IMPORTANCE
feature_importance_df = pd.DataFrame({
    "feature": X_train_final.columns,
    "importance": best_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\n" + "="*55)
print("FEATURE IMPORTANCE")
print("="*55)
print(feature_importance_df)

print("\nTop 10 Most Important Features:")
print(feature_importance_df.head(10))



FEATURE IMPORTANCE
                   feature  importance
0            car_age_years    0.236731
10          brand_Mercedes    0.153035
14      fuel_type_Electric    0.150819
1               mileage_km    0.125353
5                brand_BMW    0.101413
3               horsepower    0.068279
15        fuel_type_Hybrid    0.057897
2            engine_size_l    0.029327
18     transmission_Manual    0.013629
17  transmission_Automatic    0.012051
4          previous_owners    0.010208
21  seller_type_Individual    0.009068
12            brand_Toyota    0.006904
19   seller_type_Certified    0.005196
16        fuel_type_Petrol    0.003936
13        fuel_type_Diesel    0.002708
9                brand_Kia    0.002640
8            brand_Hyundai    0.002524
7              brand_Honda    0.002347
20      seller_type_Dealer    0.002297
6               brand_Ford    0.001820
11            brand_Nissan    0.001819

Top 10 Most Important Features:
                   feature  importance
0          